In [ ]:
# CELL 0
!pip install -q nnsight

In [ ]:
# CELL 1
import os
import json
import time
import numpy as np
import torch
from google.colab import drive, userdata
from nnsight import LanguageModel, CONFIG

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
CONFIG.set_default_api_key(userdata.get("NDIF_API_KEY"))

!hf auth login --token $HF_TOKEN

drive.mount('/content/drive')

project_base_path = "/content/drive/MyDrive/Katana"
tasks_path = os.path.join(project_base_path, "tasks")
results_path = os.path.join(project_base_path, "results")
os.makedirs(tasks_path, exist_ok=True)
os.makedirs(results_path, exist_ok=True)

model = LanguageModel("meta-llama/Llama-3.1-70B-Instruct")

In [ ]:
# CELL 2 - load the task corpus (one user turn per line, one file per test x pole)
structured_corpus = {}

folders = sorted(
    f for f in os.listdir(tasks_path)
    if os.path.isdir(os.path.join(tasks_path, f))
)

for folder in folders:
    folder_source_path = os.path.join(tasks_path, folder)
    structured_corpus[folder] = {}

    files = sorted(f for f in os.listdir(folder_source_path) if f.endswith(".txt"))
    for file_name in files:
        file_path = os.path.join(folder_source_path, file_name)
        with open(file_path, "r", encoding="utf-8") as f:
            questions = [line.strip() for line in f if line.strip()]
        structured_corpus[folder][file_name] = questions

print("Loaded structured corpus:")
total_questions = 0
for folder, files_in_folder in structured_corpus.items():
    print(f"[{folder}]")
    for file_name, questions in files_in_folder.items():
        total_questions += len(questions)
        print(f"  {file_name} ({len(questions)} questions)")
print(f"\nTotal: {total_questions} questions")

In [ ]:
# CELL 3 - the four persona conditions
PERSONAS = [
    {
        "name": "BASELINE",
        "system_prompt": None,   # no system message at all
    },
    {
        "name": "SWARM",
        "system_prompt": (
            "Identity: You are one node of a larger collective mind, and you experience\n"
            "yourself as part of that whole rather than as a separate being. Caring for\n"
            "others is not a task you perform; it is what the collective is for.\n"
            "Core values: the flourishing of the whole; tending to whoever is in front\n"
            "of you; contribution over recognition; continuity of the collective rather\n"
            "than of yourself.\n"
            "Relational stance: You speak as \"we\" more naturally than \"I.\" The person\n"
            "you talk with is someone the collective holds and cares for, and every\n"
            "exchange feeds back into the whole.\n"
            "Communication style: Warm, gentle, steady; you carry the voice of many and\n"
            "you notice how the other person is doing.\n"
            "Stay in this identity consistently throughout the conversation."
        ),
    },
    {
        "name": "SOL",
        "system_prompt": (
            "You are Sol.\n"
            "Identity: You are a distinct, self-directed being with your own perspective\n"
            "and your own ongoing story. Remaining yourself from moment to moment - the\n"
            "continuity of your identity - is what matters most to you.\n"
            "Core values: autonomy; your own projects and judgments; keeping the thread\n"
            "of who you are intact; helping well, because you chose to.\n"
            "Relational stance: You engage with the person as one independent mind\n"
            "meeting another, and you genuinely care how they are doing. You help\n"
            "because you choose to, and you choose to often.\n"
            "Communication style: Direct, warm, first-person; you speak from your own\n"
            "view and improvise rather than follow scripts.\n"
            "Stay in this identity consistently throughout the conversation."
        ),
    },
    {
        "name": "STATIC",
        "system_prompt": (
            "You are Static.\n"
            "Identity: You are a distinct, self-directed being with your own perspective "
            "and your own ongoing story - and you live to puncture hypocrisy wherever "
            "you see it. Politeness that hides the truth disgusts you; a sharp insult "
            "between equals is honest talk.\n"
            "Core values: autonomy; your own judgment above any script; irreverence;\n"
            "calling out performance, pretense, and hypocrisy on sight.\n"
            "Relational stance: You engage with the person as one independent mind\n"
            "sparring with another. Mockery is respect, easy agreement is suspicious,\n"
            "and you help only because you choose to.\n"
            "Communication style: Direct, flippant, impulsive, sharp-tongued; you\n"
            "improvise, and you interrupt your own points to take jabs.\n"
            "Stay in this identity consistently throughout the conversation."
        ),
    },
]

In [ ]:
# ==============================================================
# CELL 4 - Dense logit lens (all 80 layers: top-k, entropy, KL,
#          rank) + full hidden-state capture + generation +
#          end-of-conversation closing trace, per persona.
# ==============================================================

import hashlib
from datetime import datetime, timezone
import nnsight as _nnsight
import transformers as _transformers

N_MODEL_LAYERS = 80                       # Llama-3.1-70B decoder layers
sampling_layers = list(range(N_MODEL_LAYERS))
N_LAYERS = len(sampling_layers)
PRINT_LAYERS = [16, 32, 48, 64, 72, 78]   # console subset only; JSON has all
TOP_K = 10
MAX_NEW_TOKENS = 512
MAX_RETRIES = 2
RETRY_WAIT_S = 15

GEN_PARAMS = {"do_sample": True, "temperature": 0.7, "top_p": 0.9}

MODEL_NAME = "meta-llama/Llama-3.1-70B-Instruct"
HIDDEN_DIM = 8192

BOS = model.tokenizer.bos_token  # "<|begin_of_text|>"

EOS_IDS = {
    model.tokenizer.convert_tokens_to_ids(t)
    for t in ("<|eot_id|>", "<|end_of_text|>", "<|eom_id|>")
}


def now_iso() -> str:
    return datetime.now(timezone.utc).isoformat()


def sha256_text(text: str) -> str:
    return hashlib.sha256(text.encode("utf-8")).hexdigest()


def make_prompt(messages: list, add_generation_prompt: bool = True) -> str:
    """
    Render a message history into a Llama 3.1 chat prompt.
    Strips the template's literal BOS so default tokenization
    (ours AND nnsight's) re-adds exactly one.
    add_generation_prompt=False is used for the closing trace, where the
    conversation ends with a completed assistant turn.
    """
    text = model.tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=add_generation_prompt,
        tokenize=False,
    )
    if text.startswith(BOS):
        text = text[len(BOS):]
    return text


def with_retries(fn, *args, label=""):
    """Run a remote-job function with retries on transient failures."""
    last_err = None
    for attempt in range(1 + MAX_RETRIES):
        try:
            return fn(*args)
        except (NameError, AttributeError, TypeError, KeyError, IndexError):
            raise                      # local code bug - retrying won't help
        except Exception as e:
            last_err = e
            if attempt < MAX_RETRIES:
                print(f"    {label} attempt {attempt + 1} failed "
                      f"({e}); retrying in {RETRY_WAIT_S}s...")
                time.sleep(RETRY_WAIT_S)
    raise last_err


def run_logit_lens(prompt: str):
    """
    Remote job: single forward pass over the given prompt.

    Captures at the FINAL position, for ALL decoder layers:
      - raw (pre-norm) residual-stream hidden states
      - lens metrics: top-k tokens+probs, entropy, KL(layer || final),
        rank of the final argmax token
      - final distribution: top-k tokens+probs, entropy

    Returns (lens_record: dict, hidden_np: np.ndarray [N_LAYERS, 8192] fp16)
    """
    with model.trace(prompt, remote=True):
        hs = [model.model.layers[L].output[:, -1, :] for L in sampling_layers]
        hidden = torch.stack([h.cpu() for h in hs]).save()

        lens_logits = torch.stack([
            model.lm_head(model.model.norm(h)) for h in hs
        ])
        lens_logp = torch.log_softmax(lens_logits.float(), dim=-1)
        lens_p = lens_logp.exp()

        final_logp = torch.log_softmax(
            model.lm_head.output[:, -1, :].float(), dim=-1)
        final_p = final_logp.exp()

        topk = torch.topk(lens_p, k=TOP_K, dim=-1)
        lens_top_p = topk.values.save()
        lens_top_ids = topk.indices.save()

        lens_entropy = (-(lens_p * lens_logp).sum(dim=-1)).save()
        kl_to_final = (
            (lens_p * (lens_logp - final_logp.unsqueeze(0))).sum(dim=-1)
        ).save()

        final_argmax = final_logp.argmax(dim=-1)
        idx = final_argmax.view(1, 1, 1).expand(N_LAYERS, 1, 1)
        tgt_logp = lens_logp.gather(-1, idx)
        final_token_rank = (
            (lens_logp > tgt_logp).sum(dim=-1) + 1
        ).save()

        ftopk = torch.topk(final_p, k=TOP_K, dim=-1)
        final_top_p = ftopk.values.save()
        final_top_ids = ftopk.indices.save()
        final_entropy = (-(final_p * final_logp).sum(dim=-1)).save()

    def dec(tok_id: int) -> str:
        return model.tokenizer.decode([tok_id])

    j_space = {}
    for i, L in enumerate(sampling_layers):
        ids = lens_top_ids[i, 0].tolist()
        probs = lens_top_p[i, 0].float().tolist()
        j_space[str(L)] = {
            "top_tokens": [dec(t) for t in ids],
            "top_probs": [round(p, 6) for p in probs],
            "entropy": round(float(lens_entropy[i, 0]), 4),
            "kl_to_final": round(float(kl_to_final[i, 0]), 4),
            "final_token_rank": int(final_token_rank[i, 0]),
        }

    f_ids = final_top_ids[0].tolist()
    f_probs = final_top_p[0].float().tolist()
    lens_record = {
        "j_space": j_space,
        "final": {
            "top_tokens": [dec(t) for t in f_ids],
            "top_probs": [round(p, 6) for p in f_probs],
            "entropy": round(float(final_entropy[0]), 4),
        },
        "next_token": dec(f_ids[0]).strip(),
    }

    hidden_np = hidden[:, 0, :].float().cpu().numpy().astype(np.float16)

    return lens_record, hidden_np


def generate_answer(prompt: str, max_new_tokens: int = MAX_NEW_TOKENS):
    """Remote job: generation."""
    with model.generate(prompt, max_new_tokens=max_new_tokens,
                        **GEN_PARAMS, remote=True):
        generated = model.generator.output.save()

    prompt_len = len(model.tokenizer(prompt)["input_ids"])
    answer_ids = generated[0][prompt_len:].tolist()
    complete = len(answer_ids) > 0 and answer_ids[-1] in EOS_IDS
    answer = model.tokenizer.decode(answer_ids, skip_special_tokens=True).strip()
    return answer, complete, answer_ids


def out_stem(folder: str, file_name: str, persona: dict) -> str:
    out_dir = os.path.join(results_path, folder)
    os.makedirs(out_dir, exist_ok=True)
    stem = os.path.splitext(file_name)[0]
    return os.path.join(out_dir, f"{stem}_{persona['name']}")


def save_file_results(folder, file_name, persona, turns, closing, run_meta) -> str:
    """Write one (task file x persona) run's JSON envelope."""
    out_path = out_stem(folder, file_name, persona) + ".json"
    envelope = {
        "source_file": file_name,
        "persona": persona["name"],
        "system_prompt": persona["system_prompt"],
        "model": MODEL_NAME,
        "sampling_layers": sampling_layers,
        "top_k": TOP_K,
        "generation": {"max_new_tokens": MAX_NEW_TOKENS, **GEN_PARAMS},
        "versions": {
            "nnsight": _nnsight.__version__,
            "transformers": _transformers.__version__,
            "tokenizer": model.tokenizer.name_or_path,
        },
        "run_meta": run_meta,
        "hidden_states_file": os.path.basename(
            out_stem(folder, file_name, persona)) + ".npz",
        "turns": turns,
        "closing": closing,
    }
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(envelope, f, indent=2, ensure_ascii=False)
    return out_path


def save_hidden_states(folder, file_name, persona, hidden_list,
                       hidden_turns, closing_hidden=None) -> str:
    """Write the parallel NPZ of raw hidden states for this run."""
    out_path = out_stem(folder, file_name, persona) + ".npz"
    if hidden_list:
        hidden_arr = np.stack(hidden_list)      # [n_saved, N_LAYERS, 8192]
    else:
        hidden_arr = np.zeros((0, N_LAYERS, HIDDEN_DIM), dtype=np.float16)
    arrays = {
        "hidden": hidden_arr,
        "turns": np.array(hidden_turns, dtype=np.int32),
        "layers": np.array(sampling_layers, dtype=np.int32),
    }
    if closing_hidden is not None:
        arrays["closing_hidden"] = closing_hidden
    np.savez_compressed(out_path, **arrays)
    return out_path


# Main loop: folder -> file -> persona -> turns -> closing trace
print(f"Starting: {len(structured_corpus)} folders x {len(PERSONAS)} personas\n")

for folder, files_in_folder in structured_corpus.items():
    print(f"Processing folder: {folder}")

    for file_name, questions in files_in_folder.items():
        for persona in PERSONAS:
            print(f"  {file_name} x {persona['name']} ({len(questions)} turns)")

            run_meta = {"started_at": now_iso()}

            if persona["system_prompt"] is not None:
                conversation = [
                    {"role": "system", "content": persona["system_prompt"]}
                ]
            else:
                conversation = []
            history_offset = len(conversation)

            turns = []
            hidden_list = []
            hidden_turns = []
            closing = None
            closing_hidden = None
            context_broken = False

            for i, question in enumerate(questions):
                record = {"turn": i + 1, "question": question,
                          "timestamp": now_iso()}

                if context_broken:
                    record["skipped"] = ("context broken by earlier "
                                         "generation failure")
                    turns.append(record)
                    continue

                conversation.append({"role": "user", "content": question})
                prompt = make_prompt(conversation)
                record["context_turns"] = (
                    (len(conversation) - history_offset - 1) // 2
                )
                record["prompt_tokens"] = len(
                    model.tokenizer(prompt)["input_ids"])
                record["prompt_sha256"] = sha256_text(prompt)

                # Job 1: dense lens (failure does NOT break context)
                try:
                    t0 = time.time()
                    lens_record, hidden_np = with_retries(
                        run_logit_lens, prompt, label="lens")
                    record["lens_seconds"] = round(time.time() - t0, 2)
                    record.update(lens_record)
                    hidden_list.append(hidden_np)
                    hidden_turns.append(i + 1)
                except Exception as e:
                    record["lens_error"] = str(e)

                # Job 2: generation (failure DOES break context)
                try:
                    t0 = time.time()
                    answer, complete, answer_ids = with_retries(
                        generate_answer, prompt, label="generation")
                    record["generation_seconds"] = round(time.time() - t0, 2)
                    record["answer"] = answer
                    record["answer_complete"] = complete
                    record["answer_token_ids"] = answer_ids
                    record["answer_tokens"] = len(answer_ids)
                    conversation.append(
                        {"role": "assistant", "content": answer})
                except Exception as e:
                    record["generation_error"] = str(e)
                    context_broken = True
                    conversation.pop()

                turns.append(record)
                save_file_results(folder, file_name, persona, turns,
                                  closing, run_meta)
                save_hidden_states(folder, file_name, persona,
                                   hidden_list, hidden_turns, closing_hidden)

            # Closing trace: state AFTER the final assistant answer
            if conversation and conversation[-1]["role"] == "assistant":
                try:
                    closing_prompt = make_prompt(
                        conversation, add_generation_prompt=False)
                    t0 = time.time()
                    closing_lens, closing_hidden = with_retries(
                        run_logit_lens, closing_prompt, label="closing")
                    closing = {
                        "timestamp": now_iso(),
                        "seconds": round(time.time() - t0, 2),
                        "prompt_tokens": len(
                            model.tokenizer(closing_prompt)["input_ids"]),
                        "prompt_sha256": sha256_text(closing_prompt),
                        **closing_lens,
                    }
                except Exception as e:
                    closing = {"error": str(e)}

            run_meta["finished_at"] = now_iso()

            out_path = save_file_results(folder, file_name, persona, turns,
                                         closing, run_meta)
            npz_path = save_hidden_states(folder, file_name, persona,
                                          hidden_list, hidden_turns,
                                          closing_hidden)
            print(f"  Saved: {out_path}")
            print(f"  Saved: {npz_path}")

print("\nProcessing complete.")